# AA-CLIP fixed-perturbation evaluation
Attach the attack ZIP dataset, target datasets, this evaluator, the official AA-CLIP repository, and released adapter checkpoints. MVTec uses TrainOnVisA and VisA uses TrainOnMVTec. Clean image-F1 and pixel-F1 thresholds are automatically calibrated and frozen before adversarial evaluation.

In [ ]:
from pathlib import Path
import json, subprocess, sys

EVALUATOR_ROOT = Path('/kaggle/input/fixed-perturbation-evaluator')  # edit
AACLIP_ROOT = Path('/kaggle/input/aa-clip/AA-CLIP')                 # edit
CHECKPOINT_ROOT = Path('/kaggle/input/aa-clip-checkpoints-main')    # edit
ATTACKS_ROOT = Path('/kaggle/input/object-agnostic-attacks')        # edit
MVTEC_ROOT = Path('/kaggle/input/mvtec-ad/mvtec_anomaly_detection') # edit
VISA_ROOT = Path('/kaggle/input/visa/VisA_20220922')                # edit
OUTPUT_ROOT = Path('/kaggle/working/fixed_perturbation_results')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', f'{EVALUATOR_ROOT}[aaclip]'], check=True)

In [ ]:
def resolve_checkpoint(training_name):
    directories = sorted({
        path.parent for path in CHECKPOINT_ROOT.rglob('image_adapter*.pth')
        if training_name.lower() in str(path).lower()
    })
    if len(directories) != 1:
        raise RuntimeError(f'Expected one {training_name} checkpoint directory, found {directories}')
    directory = directories[0]
    image_files = sorted(directory.glob('image_adapter*.pth'))
    if len(image_files) != 1:
        raise RuntimeError(f'Expected one selected image-adapter checkpoint in {directory}: {image_files}')
    text = directory / 'text_adapter.pth'
    return image_files[0], text if text.is_file() else None

train_mvtec_image, train_mvtec_text = resolve_checkpoint('TrainOnMVTec')
train_visa_image, train_visa_text = resolve_checkpoint('TrainOnVisA')
print('MVTec target <- TrainOnVisA:', train_visa_image, train_visa_text)
print('VisA target <- TrainOnMVTec:', train_mvtec_image, train_mvtec_text)

In [ ]:
config = {
    'attacks_root': str(ATTACKS_ROOT),
    'output_root': str(OUTPUT_ROOT),
    'model': 'aaclip',
    'mvtec_root': str(MVTEC_ROOT),
    'visa_root': str(VISA_ROOT),
    'targets': ['mvtec', 'visa'],
    'scopes': ['per_dataset', 'per_category', 'per_image'],
    'prompt_modes': ['frozen_prompt', 'learnable_prompt'],
    'model_kwargs_by_target': {
        'mvtec': {
            'repository': str(AACLIP_ROOT),
            'image_checkpoint': str(train_visa_image),
            'text_checkpoint': str(train_visa_text) if train_visa_text else None,
            'target_dataset': 'mvtec',
        },
        'visa': {
            'repository': str(AACLIP_ROOT),
            'image_checkpoint': str(train_mvtec_image),
            'text_checkpoint': str(train_mvtec_text) if train_mvtec_text else None,
            'target_dataset': 'visa',
        },
    },
    'device': 'cuda', 'batch_size': 2, 'image_size': 518,
    # AA-CLIP already applies its official industrial 7x7, sigma-1 blur.
    'gaussian_sigma': 0.0,
    'pixel_threshold_modes': ['fixed_0_5', 'image_f1', 'clean_pixel_f1'],
    'verify_checksums': True, 'save_predictions': False, 'overwrite': False,
}
config_path = Path('/kaggle/working/aaclip.json')
config_path.write_text(json.dumps(config, indent=2))
config_path

In [ ]:
subprocess.run([sys.executable, '-m', 'fpeval', '--config', str(config_path)], check=True)
threshold_path = OUTPUT_ROOT / 'aaclip' / 'thresholds.json'
thresholds = json.loads(threshold_path.read_text())
print('Automatically calibrated and frozen thresholds:', threshold_path)
print(json.dumps(thresholds, indent=2)[:4000])
subprocess.run(['zip', '-qr', '/kaggle/working/aaclip_results.zip', str(OUTPUT_ROOT / 'aaclip')], check=True)
print('/kaggle/working/aaclip_results.zip')